In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
# from ..src.convert_pangaea_textfile import align_cropland_columns
from src.alignment import align_cropland_columns

%load_ext autoreload
%autoreload 2

# 2. Rewrite the .tab s
In order to succsessfully read in the datasets the discripbtion at the start of each .tab -file, the text-block (starting with '/*' ending with '*/') has to be deleted

In [18]:
tap_paths = ['../data/raw/pangaea_987298/datasets/N2o_ssa_cropland.tab',
             '../data/raw/pangaea_987298/datasets/N2o_ssa_forests.tab',
             '../data/raw/pangaea_987298/datasets/N2o_ssa_grasslands.tab']
csv_paths = ['../data/raw/n2o_ssa_cropland.csv',
             '../data/raw/n2o_ssa_forests.csv',
             '../data/raw/n2o_ssa_grass.csv']

for file_path, csv_path in zip(tap_paths, csv_paths):
    with open(file_path, 'r') as file:
        file_content = file.read()
    file_content = file_content[file_content.index('*/')+3:]
    # file_content = file_content.replace('\t', ',')
    file_name = Path(csv_path)
    with open(file_name, 'w') as file:
        file.write(file_content)



# 3. Read in data

In [19]:
df_cropland = pd.read_csv(csv_paths[0], sep='\t', encoding= 'utf-8')
df_forests = pd.read_csv(csv_paths[1], sep='\t', encoding= 'utf-8')
df_grasslands = pd.read_csv(csv_paths[2], sep='\t', encoding= 'utf-8')
# df_africa = pd.read_csv('../data/processed/n2o_africa_clean.csv')

## Compare column structure

Before combining the datasets, the column names are compared across land-use types.  
This is necessary because the cropland file uses more explicit column names for some ERA5 soil variables.

In [ ]:
df_cropland

In [20]:
if 'Site info' in df_cropland.columns:
    print('df_cropland del "Site info"')
    df_cropland = df_cropland.drop(columns=['Site info'])
if 'Site info' in df_forests.columns:
    print('df_cropland del "Site info"')
    df_forests = df_forests.drop(columns=['Site info'])
if 'Site info' in df_grasslands.columns:
    print('df_grasslands del "Site info"')
    df_grasslands = df_grasslands.drop(columns=['Site info'])


print(f'df_cropland.columns {len(df_cropland.columns)}')
print(f'df_forests.columns {len(df_forests.columns)}')
print(f'df_grasslands.columns {len(df_grasslands.columns)}')
# print(f'df_africa.columns {len(df_africa.columns)}')

df_cols = pd.DataFrame({
    'df_cropland' : df_cropland.columns,
    'df_forests' : df_forests.columns ,
    'df_grasslands' : df_grasslands.columns,
    # 'df_africa' : df_africa.columns
})
df_cols["cropland_diff_forests"] = df_cols["df_cropland"].ne(df_cols["df_forests"])
df_cols["cropland_diff_grasslands"] = df_cols["df_cropland"].ne(df_cols["df_grasslands"])
df_cols["forests_diff_grasslands"] = df_cols["df_forests"].ne(df_cols["df_grasslands"])

# df_cols["africa_diff_forests"] = df_cols["df_africa"].ne(df_cols["df_forests"])
# df_cols["africa_diff_cropland"] = df_cols["df_africa"].ne(df_cols["df_cropland"])
# df_cols["africa_diff_grasslands"] = df_cols["df_africa"].ne(df_cols["df_grasslands"])

df_cols_diff = df_cols[df_cols.filter(like="_diff_").any(axis=1)]
df_cols_diff

# display(df_africa.iloc[:, 11:16])
# display(df_cropland.iloc[:, 11:16])
# display(df_forests.iloc[:, 11:16])
# display(df_grasslands.iloc[:, 11:16])

df_cropland del "Site info"
df_cropland.columns 27
df_forests.columns 27
df_grasslands.columns 27


,df_cropland,df_forests,df_grasslands,cropland_diff_forests,cropland_diff_grasslands,forests_diff_grasslands
11,Soil moisture [m**3/m**3] (Content at 7-28 cm ...,Soil moisture [m**3/m**3] (ERA5 reanalyses),Soil moisture [m**3/m**3] (ERA5 reanalyses),True,True,False
12,Soil moisture [m**3/m**3] (Content at 28-100 c...,Soil moisture [m**3/m**3] (ERA5 reanalyses).1,Soil moisture [m**3/m**3] (ERA5 reanalyses).1,True,True,False
13,"T soil day m [°C] (At 0-7 cm depth, ERA5 reana...",T soil day m [°C] (ERA5 reanalyses),T soil day m [°C] (ERA5 reanalyses),True,True,False
14,"T soil day m [°C] (At 7-28 cm depth, ERA5 rean...",T soil day m [°C] (ERA5 reanalyses).1,T soil day m [°C] (ERA5 reanalyses).1,True,True,False
15,"T soil day m [°C] (At 28-100 cm depth, ERA5 re...",T soil day m [°C] (ERA5 reanalyses).2,T soil day m [°C] (ERA5 reanalyses).2,True,True,False


## Align cropland column names

The cropland dataset contains more detailed names for selected soil moisture and soil temperature columns.  
These columns are renamed to match the naming convention used in the forest and grassland datasets.

In [21]:
df_cropland = align_cropland_columns(df_cropland)


## Combine land-use datasets

After aligning the column names, the three land-use datasets are concatenated row-wise into a single dataframe.

In [22]:
df_cols = pd.DataFrame({
    'df_cropland' : df_cropland.columns,
    'df_forests' : df_forests.columns ,
    'df_grasslands' : df_grasslands.columns,
    # 'df_africa' : df_africa.columns
})
df_cols["cropland_diff_forests"] = df_cols["df_cropland"].ne(df_cols["df_forests"])
df_cols["cropland_diff_grasslands"] = df_cols["df_cropland"].ne(df_cols["df_grasslands"])
df_cols["forests_diff_grasslands"] = df_cols["df_forests"].ne(df_cols["df_grasslands"])

# df_cols["africa_diff_forests"] = df_cols["df_africa"].ne(df_cols["df_forests"])
# df_cols["africa_diff_cropland"] = df_cols["df_africa"].ne(df_cols["df_cropland"])
# df_cols["africa_diff_grasslands"] = df_cols["df_africa"].ne(df_cols["df_grasslands"])

df_cols_diff = df_cols[df_cols.filter(like="_diff_").any(axis=1)]
df_cols_diff

,df_cropland,df_forests,df_grasslands,cropland_diff_forests,cropland_diff_grasslands,forests_diff_grasslands


## Save interim dataset

The combined dataset is saved as an interim file because only column alignment and concatenation have been applied.  
Further cleaning steps are performed later.

In [23]:
df_all = pd.concat(
    [df_forests, df_cropland, df_grasslands],
    ignore_index=True,
    join="outer"
)
df_all.describe()

,Latitude,Longitude,Fert N [kg/ha] (Recorded at time of application),TTT day m [°C] (ERA5 reanalyses),TxTxTx day max [°C] (ERA5 reanalyses),TnTnTn day min [°C] (ERA5 reanalyses),Precip day total [mm/day] (ERA5 reanalyses),"Soil moisture [m**3/m**3] (Content at 0-7 cm depth, ERA5...)",Soil moisture [m**3/m**3] (ERA5 reanalyses),Soil moisture [m**3/m**3] (ERA5 reanalyses).1,...,VPD day m [kPa] (ERA5 reanalyses),PPPP day m [hPa] (ERA5 reanalyses),SWD day m [W/m**2] (ERA5 reanalyses),PPFD day m [µmol/m**2/s] (ERA5 reanalyses),Duration [days] (Since last precipitation even...),Duration [days] (Since last fertiliser applica...),Fert N dec adj exp [kg/ha] (Exponential decay model (k=0.05)),Transformation S (Modeled),Transformation C (Modeled),"N2O flux [µg/m**2/h] (From soil surface, Modeled)"
count,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,...,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000,5280.000000
mean,-1.228231,31.331031,38.504735,21.133681,25.754478,17.674016,1.489663,0.315828,0.310683,0.294414,...,0.677221,864.763174,38.194886,77.153669,3.509848,20.852083,15.123450,0.130964,-0.123367,12.095709
std,3.703325,10.765475,46.788347,3.171982,3.514496,3.438315,4.270444,0.106214,0.105178,0.098814,...,0.334093,59.804719,9.087886,18.357529,20.028591,37.419433,30.809558,0.690452,0.700777,20.839097
min,-17.706000,-0.381800,0.000000,13.069766,16.074131,6.959682,0.000000,0.078000,0.078000,0.066000,...,0.049908,789.017518,2.077375,4.196297,0.000000,0.000000,0.000000,-1.000000,-1.000000,-21.149615
25%,-2.139900,32.692700,0.000000,18.666322,22.958315,15.252268,0.001318,0.225582,0.225204,0.219140,...,0.436225,829.737725,32.366683,65.380701,0.000000,0.000000,0.000000,-0.528964,-0.809017,1.896372
50%,-1.260000,35.380000,45.000000,20.614597,25.779030,16.875000,0.130284,0.328086,0.316613,0.300809,...,0.606044,833.091315,38.207133,77.178409,0.000000,2.000000,0.000000,0.283206,-0.283206,5.142646
75%,0.321900,36.720000,50.000000,23.190000,28.127880,19.555776,1.126393,0.401567,0.398310,0.370087,...,0.852115,895.864040,44.725212,90.344926,0.000000,33.000000,14.505386,0.790776,0.557324,13.045997
max,6.054500,38.226900,200.000000,28.942652,35.700000,26.212832,69.000000,0.507654,0.506686,0.510832,...,1.954649,994.700000,59.174045,119.531572,205.000000,251.000000,194.272423,1.000000,1.000000,242.346036


In [24]:
out_dir = Path("../data/interim")
out_dir.mkdir(parents=True, exist_ok=True)

df_all.to_csv(
    out_dir / "n2o_ssa_landuse_aligned_combined.csv",
    index=False
)